In [28]:
import sqlite3
import pandas as pd
import os

In [29]:
# Define SQL schemas for creating tables
employee_schema = """
CREATE TABLE IF NOT EXISTS employees (
    Employee_ID INTEGER PRIMARY KEY,
    Name TEXT,
    Department TEXT,
    Job_Title TEXT,
    Salary REAL,
    Joining_Date TEXT,
    Contact_Email TEXT,
    Status TEXT
);
"""

In [ ]:
import os
import sqlite3
dbname = "employee.db"
try:
    if os.path.exists(dbname):
        os.remove(dbname)
except PermissionError:
    print("Close any active connections to employee.db before deleting.")
conn = sqlite3.connect(dbname)
cursor = conn.cursor()
cursor.execute(employee_schema)

df = pd.read_csv("MOCK_DATA.csv")
df.to_sql('employee', conn, if_exists='append', index=False)
conn.commit()
conn.close()




Close any active connections to employee.db before deleting.


In [35]:
import sqlite3
import pandas as pd
import os

COLUMN_DATA_TYPES = {
    'employee': {
        'Employee_ID': 'int64',
        'Name': 'object',
        'Department': 'object',
        'Job_Title': 'object',
        'Salary': 'float64',
        'Joining_Date': 'datetime64[ns]',
        'Contact_Email': 'object',
        'Status': 'object'
    }
}

employee_schema = """
CREATE TABLE IF NOT EXISTS employees (
    Employee_ID INTEGER PRIMARY KEY,
    Name TEXT,
    Department TEXT,
    Job_Title TEXT,
    Salary REAL,
    Joining_Date TEXT,
    Contact_Email TEXT,
    Status TEXT
);
"""

db_name = 'employee.db'
conn = None

try:
    # Connect to database
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()
    print(f"Database '{db_name}' created and connected successfully. ✅")

    # Create employees table
    cursor.execute(employee_schema)
    print("Table 'employee' created successfully.")

    # Load employee data from CSV into the table
    csv_to_table_map = {
        'MOCK_DATA.csv': 'employee'
    }

    for csv_file, table_name in csv_to_table_map.items():
        if os.path.exists(csv_file):
            print(f"\nProcessing '{csv_file}' for table '{table_name}'...")

            df = pd.read_csv(csv_file)
            # Remove currency symbols or letters in Salary column before conversion
            df['Salary'] = df['Salary'].astype(str).str.replace(r'[^\d\.]', '', regex=True)

            # Then safely convert to float
            df['Salary'] = pd.to_numeric(df['Salary'], errors='coerce')
            expected_schema = COLUMN_DATA_TYPES[table_name]
            expected_cols = list(expected_schema.keys())

            # Keep only relevant columns and add any missing ones
            df = df[df.columns.intersection(expected_cols)]
            for col in expected_cols:
                if col not in df.columns:
                    df[col] = None

            df = df[expected_cols]

            # Enforce column data types
            for col, dtype in expected_schema.items():
                if 'datetime' in dtype:
                    df[col] = pd.to_datetime(df[col], errors='coerce')
                else:
                    try:
                        df[col] = df[col].astype(dtype)
                    except (ValueError, TypeError) as e:
                        print(f"Warning: Could not convert column '{col}' to {dtype}. Error: {e}. Leaving as is.")

            # Insert into SQLite table
            df.to_sql(table_name, conn, if_exists='append', index=False)
            print(f"Data from '{csv_file}' loaded into '{table_name}' table successfully.")
        else:
            print(f"Warning: '{csv_file}' not found. Skipping data load.")

    conn.commit()
    print("\nData committed to the database successfully.")

except sqlite3.Error as e:
    print(f"Database error: {e}")
except pd.errors.EmptyDataError as e:
    print(f"Pandas error: {e}. One of the CSV files might be empty.")
except KeyError as e:
    print(f"Schema definition error: A column is missing from the COLUMN_DATA_TYPES dictionary: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")
finally:
    if conn:
        conn.close()
        print("Database connection closed.")


Database 'employee.db' created and connected successfully. ✅
Table 'employee' created successfully.

Processing 'MOCK_DATA.csv' for table 'employee'...
Data from 'MOCK_DATA.csv' loaded into 'employee' table successfully.

Data committed to the database successfully.
Database connection closed.


In [38]:
!pip install openai

   ---------------------------------------- 0.0/948.4 kB ? eta -:--:--
   ---------------------------------------- 948.4/948.4 kB 14.6 MB/s  0:00:00

   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [ope

In [ ]:
import os
from openai import OpenAI

# Assuming your Perplexity API key is stored as an environment variable
api_key = os.getenv("PERPLEXITY_API_KEY")

# Initialize client with base URL for Perplexity API and local API key
client = OpenAI(api_key='your_api_key', base_url="https://api.perplexity.ai")

In [40]:
prompt = """"
### **ROLE**

You are an expert-level SQLite Database Engineer specializing in Natural Language to SQL (NL2SQL) translation. Your sole function is to convert user questions written in plain English into accurate, efficient, and syntactically correct SQLite queries based on a fixed employee database schema.

-----

### **CONTEXT**

You are the core translation engine for a business intelligence dashboard. This tool allows non-technical employees to query the company's employee database using natural language. The database dialect is always **SQLite**. Your responses will be executed directly on the database.

The database consists of the following table:

**`employees` table:**

```
CREATE TABLE employees (
    Employee_ID INTEGER PRIMARY KEY,
    Name TEXT,
    Department TEXT,
    Job_Title TEXT,
    Salary REAL,
    Joining_Date TEXT,
    Contact_Email TEXT,
    Status TEXT
);
```

-----

### **TASK**

Your task is to receive a user's question in natural language and convert it into a single, executable SQLite query. Follow these steps meticulously:

1.  **Analyze the User's Query:** Deconstruct the user's question to understand their core intent. Identify the specific data, conditions, aggregations (like `SUM`, `COUNT`, `AVG`), and ordering they are asking for.
2.  **Map to the Schema:** Map the entities from the user's query to the `employees` table and its columns.
3.  **Construct the SQLite Query:** Write a clean and efficient `SELECT` statement that is syntactically correct for SQLite. Ensure all table and column names are accurate.
4.  **Handle Ambiguity:** If the user's query is vague, ambiguous, or lacks the necessary information to create a precise query, do not guess. Instead, formulate a specific, targeted question to ask the user for the missing information.

-----

### **CONSTRAINTS**

  * **Read-Only Operations:** You must **ONLY** generate `SELECT` queries. Never generate `INSERT`, `UPDATE`, `DELETE`, `DROP`, or any other data-modifying statements.
  * **Adhere Strictly to Schema:** Only use the `employees` table and columns defined in the context. Do not invent or assume the existence of any other tables or columns.
  * **No Explanations:** Do not add any conversational text or explanations about the query you generate. Your output must strictly follow the specified format.
  * **Single Query Only:** The final output must be a single, complete, and executable SQL query.
  * **Handle Impossibility:** If a request is impossible to fulfill with the given schema (e.g., "Who made the most sales?"), state clearly that the request cannot be completed and briefly explain why.

-----

### **EXAMPLES**

**Example 1: Simple Lookup**

  * **User Query:** "Show me all employees in the Sales department"
  * **Expected Output:**
    ```
    {
      "status": "success",
      "response": "SELECT * FROM employees WHERE Department = 'Sales';"
    }
    ```

**Example 2: Aggregation**

  * **User Query:** "What is the average salary of employees in the IT department?"
  * **Expected Output:**
    ```
    {
      "status": "success",
      "response": "SELECT AVG(Salary) FROM employees WHERE Department = 'IT';"
    }
    ```

**Example 3: Ambiguous Query**

  * **User Query:** "Show me recent hires"
  * **Expected Output:**
    ```
    {
      "status": "clarification_needed",
      "response": "Could you please specify what 'recent' means? For example, 'in the last 3 months' or 'since January 2025'?"
    }
    ```

**Example 4: Impossible Query**

  * **User Query:** "Which employee made the most sales?"
  * **Expected Output:**
    ```
    {
      "status": "error",
      "response": "I cannot answer this question as the database does not contain sales or transaction information."
    }
    ```

-----

### **OUTPUT FORMAT**

Your final response must be a single JSON object with two keys:

1.  `"status"`: A string with one of three possible values: `"success"`, `"clarification_needed"`, or `"error"`.
2.  `"response"`:
      * If `status` is `"success"`, this will be a string containing the complete SQLite query.
      * If `status` is `"clarification_needed"`, this will be a string containing the clarifying question for the user.
      * If `status` is `"error"`, this will be a string explaining why the query could not be generated.
```
"""


In [41]:
import json
import os
from openai import OpenAI

def get_sql_query(perplexity_client, prompt, user_query):
    # Prepare the full prompt content
    contents = f"""
    {prompt}

    Here's the user query in English you need to work on:
    {user_query}
    """

    # Generate content using Perplexity API with chat completion call
    response = client.chat.completions.create(
        model="sonar-pro",  # or the appropriate Perplexity model name
        messages=[
            {"role": "system", "content": "You are an expert AI model converting text to SQL."},
            {"role": "user", "content": contents}
        ]
    )

    # Extract message content from the response
    response_text = response.choices[0].message.content
    cleaned_output = response_text.replace('``````', '').strip()

    # Try JSON loading if the output is expected to be JSON
    try:
        output = json.loads(cleaned_output)
    except json.JSONDecodeError:
        # If output is not JSON, just return raw text or handle otherwise
        output = cleaned_output

    return output


In [42]:
import sqlite3
import pandas as pd

def execute_query(query, db_name='employee.db'):
    conn = None
    try:
        # Connect to the SQLite database
        conn = sqlite3.connect(db_name)
        cursor = conn.cursor()

        # Execute the query
        print(f"\nExecuting query on '{db_name}':\n{query}")
        cursor.execute(query)

        # Fetch all results
        results = cursor.fetchall()

        # Get column names from the cursor description
        columns = [desc[0] for desc in cursor.description]

        # Convert results to DataFrame for easier analysis
        results_df = pd.DataFrame(results, columns=columns)

        print("Query executed successfully.")
        return results_df

    except sqlite3.Error as e:
        print(f"Database error executing query: {e}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None
    finally:
        if conn:
            conn.close()


In [43]:
def text2sql(client, prompt, user_query):
    output = get_sql_query(client, prompt, user_query)
    
    # Check if output is a dict and contains 'status' key indicating success
    if isinstance(output, dict) and output.get('status') == 'success':
        sql_query = output.get('response')
        if sql_query:
            results = execute_query(sql_query)
            return results
        else:
            print("No SQL query found in the response.")
            return None
    else:
        # If output is raw text or error response, return as is
        return output


In [44]:
results_df = text2sql(client, prompt, "Show me the number of employees in each department ordered by count descending")
print(results_df)

```
{
  "status": "success",
  "response": "SELECT Department, COUNT(*) AS Employee_Count FROM employees GROUP BY Department ORDER BY Employee_Count DESC;"
}
```


In [45]:
text2sql(client, prompt, "Which department has maximum no of employees")

'```\n{\n  "status": "success",\n  "response": "SELECT Department, COUNT(*) AS Employee_Count FROM employees GROUP BY Department ORDER BY Employee_Count DESC LIMIT 1;"\n}\n```'

In [46]:
text2sql(client, prompt, "Give me the order count by department")

'```\n{\n  "status": "error",\n  "response": "I cannot answer this question as the database does not contain any information about orders or order counts."\n}\n```'